# Step 3 – NLP & RAG: Intelligent Quality Audit Reports & Operator Copilot

M.Sc. team project – Step 3 of the 4-step inspection pipeline:
```
vision (step 1)  ──►  RL scheduling (step 2)  ──►  NLP / RAG reports (step 3)  ──►  BI dashboard (step 4)
```

### Overview
This component implements a **Retrieval-Augmented Generation (RAG) and Quality Auditing System** that translates numeric vision scores and RL scheduling actions into **actionable manufacturing intelligence**:

1. **Manufacturing Knowledge Base**: Ingests and indexes standard operating procedures (SOPs), defect taxonomies, and machine maintenance guidelines for MVTec categories (`grid`, `zipper`, `hazelnut`, `leather`, `toothbrush`).
2. **Telemetry Fusion**: Ingests telemetry outputs from Step 1 (`current_anomaly_score.csv`) and Step 2 (`scheduler_inference_ready.csv`, `adaptation_demo_results.csv`).
3. **Automated Incident & Shift Reports**: Generates executive audit reports detailing throughput, RL cost savings vs. 100% inspection, and SOP-grounded remediation plans for defect clusters.
4. **Interactive Operator Copilot**: Provides floor technicians with an interactive assistant that explains why specific parts were escalated by RL and prescribes exact rework/scrap protocols citing official SOP clauses.


## 1. Environment Setup & Configuration

This section configures Google Colab Drive access (or local repository fallback) and installs optional dependencies. Colab already includes standard scientific libraries.


In [ ]:
# Check environment and install extras if running in Colab
import sys
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab environment.")
except ImportError:
    IN_COLAB = False
    print("Running in local environment.")

# Optional: sentence-transformers for state-of-the-art dense semantic embeddings
try:
    import sentence_transformers
except ImportError:
    print("Installing sentence-transformers...")
    !pip install -q sentence-transformers


In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
import numpy as np

# ======================= CONFIG =======================
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Integrated')
    OUTPUTS_DIR = BASE_DIR / 'outputs'
else:
    REPO_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
    BASE_DIR = REPO_ROOT
    OUTPUTS_DIR = REPO_ROOT / 'outputs'

VISION_OUTPUTS_DIR = OUTPUTS_DIR / 'vision'
SCHEDULER_OUTPUTS_DIR = OUTPUTS_DIR / 'scheduler_outputs'
RAG_OUTPUTS_DIR = OUTPUTS_DIR / 'rag_outputs'
RAG_OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Vision outputs:", VISION_OUTPUTS_DIR)
print("Scheduler outputs:", SCHEDULER_OUTPUTS_DIR)
print("RAG outputs:", RAG_OUTPUTS_DIR)


## 2. Manufacturing Knowledge Base Ingestion

We ingest authoritative Standard Operating Procedures (SOPs) created for the inspection system. Each document contains:
- Defect taxonomies, visual signatures, and severity ratings.
- Mechanical and thermal root causes.
- Rework protocols vs. Scrap/Disposition criteria.
- Machine maintenance checklists.

Documents are semantically chunked and tagged with structured metadata (`category`, `defect_code`, `doc_id`, `section_title`).


In [ ]:
import re

def parse_sop_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    doc_id = 'SOP'
    category = 'general'

    doc_id_match = re.search(r"-\s*\*\*Document ID\*\*:\s*([^\n\r]+)", content)
    if doc_id_match:
        doc_id = doc_id_match.group(1).strip()
    else:
        doc_id_hdr = re.search(r"^#\s*([A-Z0-9\-]+):", content, re.MULTILINE)
        if doc_id_hdr:
            doc_id = doc_id_hdr.group(1).strip()

    cat_match = re.search(r"-\s*\*\*Category\*\*:\s*`?([a-zA-Z0-9_\-]+)`?", content)
    if cat_match:
        category = cat_match.group(1).strip().lower()

    sections = re.split(r"(?=(?:^|\n)#{2,3}\s+)", content)
    chunks = []

    for sec in sections:
        sec = sec.strip()
        if not sec:
            continue

        hdr_match = re.match(r"^#{2,3}\s+([^\n]+)", sec)
        header = hdr_match.group(1).strip() if hdr_match else f"Overview ({doc_id})"

        defect_code = 'general'
        defect_match = re.search(r"Defect Code:\s*`?([a-zA-Z0-9_\-]+)`?", header, re.IGNORECASE)
        if defect_match:
            defect_code = defect_match.group(1).strip().lower()

        chunks.append({
            'doc_id': doc_id,
            'category': category,
            'defect_code': defect_code,
            'section_title': header,
            'content': sec,
            'source_file': Path(file_path).name
        })

    return chunks

# Locate knowledge base
kb_dir = REPO_ROOT / 'rag' / 'knowledge_base' if not IN_COLAB else Path('/content/Adaptive_Visual_Inspection-_System/rag/knowledge_base')
if not kb_dir.exists():
    kb_dir = Path('../rag/knowledge_base')

chunks = []
for p in sorted(kb_dir.glob('*.md')):
    chunks.extend(parse_sop_file(p))

print(f"Successfully loaded and chunked {len(chunks)} SOP sections from {kb_dir.name}/")
pd.DataFrame([{'doc_id': c['doc_id'], 'category': c['category'], 'defect_code': c['defect_code'], 'title': c['section_title'][:40]} for c in chunks[:8]])


## 3. Vector Embeddings & Knowledge Indexing

Chunks are encoded into high-dimensional vector representations. If `sentence-transformers` is installed, dense embeddings (`all-MiniLM-L6-v2`) are computed; otherwise, the resilient built-in subword semantic encoder is employed.


In [ ]:
class SimpleVectorStore:
    def __init__(self):
        self.chunks = []
        self.embeddings = None

    def add(self, chunks, embeddings):
        self.chunks = chunks
        self.embeddings = np.array(embeddings, dtype=np.float32)

    def query(self, query_vec, category=None, defect_code=None, top_k=3):
        q_norm = np.linalg.norm(query_vec)
        if q_norm > 0:
            query_vec = query_vec / q_norm
        scores = np.dot(self.embeddings, query_vec)

        candidates = []
        for i, c in enumerate(self.chunks):
            if category and c.get('category') not in [category, 'general']:
                continue
            if defect_code and c.get('defect_code') not in [defect_code, 'general']:
                continue
            candidates.append(i)

        if not candidates and category:
            candidates = [i for i, c in enumerate(self.chunks) if c.get('category') in [category, 'general']]
        if not candidates:
            candidates = list(range(len(self.chunks)))

        filtered_scores = scores[candidates]
        ranked_indices = np.argsort(filtered_scores)[::-1][:top_k]

        return [
            {**self.chunks[candidates[r]], 'similarity_score': float(filtered_scores[r])}
            for r in ranked_indices
        ]

# Compute embeddings
try:
    from sentence_transformers import SentenceTransformer
    embed_model = SentenceTransformer('all-MiniLM-L6-v2')
    chunk_vectors = embed_model.encode([c['content'] for c in chunks], normalize_embeddings=True)
    def encode_query(q):
        return embed_model.encode([q], normalize_embeddings=True)[0]
    print("Using SentenceTransformers ('all-MiniLM-L6-v2') dense embeddings.")
except Exception as e:
    print(f"Using built-in feature encoder: {e}")
    def encode_query(q, dim=384):
        vec = np.zeros(dim, dtype=np.float32)
        for t in re.findall(r'\b\w+\b', q.lower()):
            vec[hash(t) % dim] += 1.0
        n = np.linalg.norm(vec)
        return vec / n if n > 0 else vec
    chunk_vectors = np.array([encode_query(c['content']) for c in chunks])

vector_store = SimpleVectorStore()
vector_store.add(chunks, chunk_vectors)
print(f"Indexed {len(chunks)} chunks into vector store.")


## 4. SOP Retrieval Verification

Let us verify that our hybrid retrieval system accurately returns the relevant standard operating procedures when queried with technical manufacturing defect terms.


In [ ]:
test_query = "rework procedure and root cause for bent grid wire"
matched = vector_store.query(encode_query(test_query), category="grid", defect_code="bent", top_k=2)

print(f"Query: '{test_query}'\n")
for i, m in enumerate(matched, 1):
    print(f"--- Result {i} (Similarity: {m['similarity_score']:.3f}) ---")
    print(f"Document ID : {m['doc_id']}")
    print(f"Section     : {m['section_title']}")
    print(f"Excerpt     :\n{m['content'][:250]}...\n")


## 5. Ingesting Step 1 (Vision) & Step 2 (RL) Telemetry

The RAG engine connects to the actual inference CSVs generated upstream:
- `scheduler_inference_ready.csv`: Decisions made by the Q-learning policy (`skip`, `prediction`, `expert`).
- `current_anomaly_score.csv`: Feature distance scores and threshold margins.
- `adaptation_demo_summary.csv`: Adaptation metrics under defect rate surge.


In [ ]:
sched_csv = SCHEDULER_OUTPUTS_DIR / 'scheduler_inference_ready.csv'
vision_csv = VISION_OUTPUTS_DIR / 'current_anomaly_score.csv'

if sched_csv.exists():
    print(f"Loading real telemetry from {sched_csv.name}...")
    telemetry_df = pd.read_csv(sched_csv)
    if vision_csv.exists():
        vdf = pd.read_csv(vision_csv)
        v_mah = vdf[vdf['method'].str.lower() == 'mahalanobis'].copy()
        telemetry_df = telemetry_df.merge(
            v_mah[['image_path', 'anomaly_score', 'normalized_score', 'predicted_defect_type']],
            on='image_path', how='left'
        )
else:
    print("Upstream scheduler CSV not found on local disk. Generating representative synthetic stream...")
    from rag.telemetry_loader import TelemetryLoader
    telemetry_df = TelemetryLoader().load_inference_telemetry()

print(f"Total operational records loaded: {len(telemetry_df)}")
telemetry_df.head()


## 6. Automated Quality & Incident Shift Report Generation

Here we synthesize operational metrics into an executive-level quality audit report. The system:
1. Calculates cost savings of the RL scheduler against the 100% manual inspection baseline.
2. Identifies high-risk defect clusters.
3. Retrieves authoritative SOP root causes and rework protocols from the vector store.
4. Saves a structured report ready for plant management and the Step 4 BI dashboard.


In [ ]:
from rag.retriever import RAGRetriever
from rag.embeddings import EmbeddingModel
from rag.report_generator import QualityReportGenerator

# Initialize report generator with our vector store
class NotebookRetriever:
    def __init__(self, vs, enc):
        self.vs = vs
        self.enc = enc
    def retrieve(self, query, category=None, defect_code=None, top_k=2):
        return self.vs.query(self.enc(query), category=category, defect_code=defect_code, top_k=top_k)
    def format_context_for_prompt(self, chunks):
        return "\n\n".join([f"[{c['doc_id']} - {c['section_title']}]\n{c['content']}" for c in chunks])

nb_retriever = NotebookRetriever(vector_store, encode_query)
report_gen = QualityReportGenerator(nb_retriever)
report_res = report_gen.generate_shift_report(shift_name="Shift 1 - Precision Line A")

print(f"Report generated successfully!")
print(f"Simulated Cost Reduction: {report_res['cost_savings_pct']:.1f}%")
print(f"Saved to: {report_res['report_path']}")
print("\n" + "="*70)
print("REPORT PREVIEW (FIRST 1200 CHARACTERS):")
print("="*70)
print(report_res['markdown_report'][:1200] + "\n...")


## 7. Interactive Operator Copilot

Floor technicians can query the copilot with specific image paths or ask general defect troubleshooting questions.


In [ ]:
from rag.copilot import OperatorCopilot

copilot = OperatorCopilot(nb_retriever)

# Test 1: Diagnose specific high-risk part
sample_img = telemetry_df[telemetry_df['risk_level'] == 'high'].iloc[0]['image_path']
print(f"Diagnosing item: {sample_img}")
diag = copilot.diagnose_image(sample_img)

print(diag['operator_guidance'])


In [ ]:
# Test 2: General floor technician query
q = "What is the scrap and salvage policy for cracked hazelnuts?"
res = copilot.ask(q, category="hazelnut")
print(res['response'])


## 8. Exporting Artifacts for Step 4 (BI Dashboard)

The report and telemetry summaries are saved in `outputs/rag_outputs/` so that Step 4 (BI Dashboard) can seamlessly render them in visual charts and operator side-panels.


In [ ]:
summary_export = {
    'total_units_processed': int(len(telemetry_df)),
    'rl_action_distribution': telemetry_df['selected_action'].value_counts().to_dict(),
    'risk_tier_distribution': telemetry_df['risk_level'].value_counts().to_dict(),
    'report_markdown_path': str(RAG_OUTPUTS_DIR / 'shift_quality_audit_report.md')
}

with open(RAG_OUTPUTS_DIR / 'rag_shift_summary.json', 'w') as f:
    json.dump(summary_export, f, indent=4)

print("Exported Step 3 artifacts:")
for p in RAG_OUTPUTS_DIR.glob('*'):
    print(f"- {p.name}")
